Our package provides data access in a Python programming environment.

Here, we will start a Clustering analysis for the Pancreatic ductal adenocarcinoma (pdac).

In [5]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from gpnotebook.tools.standard_imports import *
import yaml


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
project = cptac_glyco.Pdac()

Load glycopeptides expression matrix.

In [7]:
meta_wd = r"D:\yhu39\github\glycoproteinnotebook-private\public_data\meta\PDAC_G_PDC000272"
path = os.path.join(meta_wd, "PDAC_samples.txt")
meta_df = pd.read_csv(path, sep="\t")
meta_df

,sample,patient,pathological_status,data_set,qualified
0,pool_01,NaN,pool,1,T
1,QC1_C_01,QC1,C,1,T
2,C3N-03884_T_01,C3N-03884,T,1,T
3,C3L-00589_N_F_01,C3L-00589,N,1,F
4,C3L-03123_N_01,C3L-03123,N,1,T
...,...,...,...,...,...
270,C3L-07032_N_25,C3L-07032,D,25,T
271,C3L-07033_N_25,C3L-07033,D,25,T
272,C3L-07034_N_25,C3L-07034,D,25,T
273,C3L-07035_N_25,C3L-07035,D,25,T


In [44]:
tumor_samples = meta_df[(meta_df["qualified"] == "T") & (meta_df["pathological_status"] == "T")]["sample"].tolist()

In [45]:
len(tumor_samples)

105

In [24]:
meta = project.get_meta_table(project="PDAC-105")
meta

,Sample.ID,Patient.ID,Project.ID,Pathological.Status,xCell,multiomics.NMF.cluster,KRAS.Mutation
0,PDA.C3N-01715.T,C3N-01715,PDAC-CPTAC3-Discovery,Tumor,B,2.0,NaN
1,PDA.C3N-03426.T,C3N-03426,PDAC-CPTAC3-Discovery,Tumor,B,2.0,NaN
2,PDA.C3N-00198.T,C3N-00198,PDAC-CPTAC3-Discovery,Tumor,NaN,1.0,NaN
3,PDA.C3N-01380.T,C3N-01380,PDAC-CPTAC3-Discovery,Tumor,NaN,1.0,NaN
4,PDA.C3N-01388.T,C3N-01388,PDAC-CPTAC3-Discovery,Tumor,NaN,2.0,p.G12D
...,...,...,...,...,...,...,...
175,PDA.C3L-07032.N,C3L-07032,PDAC-CPTAC3-Discovery,Normal-duct,NaN,NaN,NaN
176,PDA.C3L-07033.N,C3L-07033,PDAC-CPTAC3-Discovery,Normal-duct,NaN,NaN,NaN
177,PDA.C3L-07034.N,C3L-07034,PDAC-CPTAC3-Discovery,Normal-duct,NaN,NaN,NaN
178,PDA.C3L-07035.N,C3L-07035,PDAC-CPTAC3-Discovery,Normal-duct,NaN,NaN,NaN


In [75]:
meta2 = meta[meta["Pathological.Status"]=="Tumor"].loc[:,["Patient.ID","xCell","multiomics.NMF.cluster","KRAS.Mutation"]]


In [26]:
meta2

,Patient.ID,xCell,multiomics.NMF.cluster,KRAS.Mutation
0,C3N-01715,B,2.0,NaN
1,C3N-03426,B,2.0,NaN
2,C3N-00198,NaN,1.0,NaN
3,C3N-01380,NaN,1.0,NaN
4,C3N-01388,NaN,2.0,p.G12D
...,...,...,...,...
100,C3L-01328,NaN,2.0,p.G12D
101,C3L-03639,B,2.0,p.G12D
102,C3N-04282,C,2.0,p.Q61H
103,C3L-04072,B,2.0,p.G12D


Load clinical matrix and create a dictionary mapping tumor patient ID and vital status(dead or alive). 

In [28]:
pdac = cptac.Pdac()
clinical = pdac.get_clinical('mssm')


In [30]:
clinical.head(2)

Name,tumor_code,discovery_study,type_of_analyzed_samples,confirmatory_study,type_of_analyzed_samples,age,sex,race,ethnicity,ethnicity_race_ancestry_identified,...,additional_treatment_pharmaceutical_therapy_for_new_tumor,additional_treatment_immuno_for_new_tumor,number_of_days_from_date_of_initial_pathologic_diagnosis_to_date_of_additional_surgery_for_new_tumor_event_loco-regional,number_of_days_from_date_of_initial_pathologic_diagnosis_to_date_of_additional_surgery_for_new_tumor_event_metastasis,"Recurrence-free survival, days","Recurrence-free survival from collection, days","Recurrence status (1, yes; 0, no)","Overall survival, days","Overall survival from collection, days","Survival status (1, dead; 0, alive)"
Patient_ID,,,,,,,,,,,,,,,,,,,,,
C3L-00017,PDA,Yes,Tumor,NaN,NaN,69,Male,White,Not Hispanic or Latino,Caucasian,...,NaN,NaN,NaN,NaN,NaN,NaN,0,426.0,426.0,0.0
C3L-00102,PDA,Yes,Tumor_and_Normal,NaN,NaN,42,Male,White,Not Hispanic or Latino,White,...,No,No,239.0,NaN,172.0,172.0,1,249.0,249.0,1.0


In [31]:

# print(clinical.columns.values)
vital_map = dict()
for index,row in clinical.iterrows():
  sample =  index 
  vital_status = row['Survival status (1, dead; 0, alive)']
  vital_map[sample] = 'NaN' if pd.isna(vital_status) else 'dead' if int(vital_status) == 1 else 'alive'

Create a top annotation dataframe including tumor patiens ID, 'xCell','multiomics.NMF.cluster','KRAS.Mutation, and vital status information.

In [34]:
meta2.head(2)

,Patient.ID,xCell,multiomics.NMF.cluster,KRAS.Mutation
0,C3N-01715,B,2.0,NaN
1,C3N-03426,B,2.0,NaN


In [76]:
meta2['Vital'] = meta2['Patient.ID'].apply(lambda x: vital_map[x] if x in vital_map else x)
meta2 = meta2.replace(np.nan, 'NaN')
meta2['multiomics.NMF.cluster'] = meta2['multiomics.NMF.cluster'].apply(lambda x: 'NaN' if x == 'NaN' else str(int(x)))

In [41]:
job_dir = r"D:\yhu39\github\glycoproteinnotebook-private\public_data\cache\PDAC_G_PDC000272\cluster"


In [61]:
meta2.head(2)

,Patient.ID,xCell,multiomics.NMF.cluster,KRAS.Mutation,Vital
0,C3N-01715,B,2,NaN,dead
1,C3N-03426,B,2,NaN,dead


In [62]:
meta_df.head(2)

,sample,patient,pathological_status,data_set,qualified
0,pool_01,NaN,pool,1,T
1,QC1_C_01,QC1,C,1,T


In [63]:
meta_df2 = meta_df[meta_df["sample"].isin(tumor_samples)]

In [65]:
meta_df2.head(2)

,sample,patient,pathological_status,data_set,qualified
2,C3N-03884_T_01,C3N-03884,T,1,T
6,C3L-03123_T_01,C3L-03123,T,1,T


In [66]:
meta_d = dict(zip(meta_df2["patient"], meta_df2["sample"]))

In [77]:
meta2['Sample.ID'] = meta2['Patient.ID'].apply(lambda x: meta_d[x] if x in meta_d else x)
meta2 = meta2.drop(columns=["Patient.ID"],axis=1)
meta2 = meta2.loc[:,['Sample.ID'] + [col for col in meta2.columns if col != 'Sample.ID']]

In [78]:
meta2

,Sample.ID,xCell,multiomics.NMF.cluster,KRAS.Mutation,Vital
0,C3N-01715_T_22,B,2,NaN,dead
1,C3N-03426_T_18,B,2,NaN,dead
2,C3N-00198_T_06,NaN,1,NaN,alive
3,C3N-01380_T_18,NaN,1,NaN,dead
4,C3N-01388_T_04,NaN,2,p.G12D,dead
...,...,...,...,...,...
100,C3L-01328_T_20,NaN,2,p.G12D,alive
101,C3L-03639_T_20,B,2,p.G12D,dead
102,C3N-04282_T_13,C,2,p.Q61H,dead
103,C3L-04072_T_16,B,2,p.G12D,dead


In [79]:
top_ann_data_path = os.path.join(job_dir,'top_ann_data.tsv')
meta2.to_csv(top_ann_data_path, sep="\t", index=False)

Top annotation settings.

In [39]:

top_ann_settings = {
    'xCell': {
        'A': 'red',
        'B': 'blue',
        'C': 'green',
        'D': 'brown',
        'NaN': 'grey'
    },
    'Vital': {
        'alive': 'blue',
        'dead': 'red',
        'NaN': 'grey'
    },
    'multiomics.NMF.cluster': {
        '1': 'red',
        '2': 'blue'
    },
    'KRAS.Mutation':{
        "p.G12D": 'red',        
        "p.G12R": 'green', 
        "p.G12V": 'blue',
        "p.G13D": 'purple',  
        "p.Q61H": 'yellow', 
        "p.Q61R": 'pink',
        "NaN": 'grey'

    }

}
top_ann_settings_path = os.path.join(job_dir,'top_ann_settings.yml')
with open(top_ann_settings_path,'w') as f:
    yaml.dump(top_ann_settings,f,default_flow_style=False)

In [42]:
wd = r"D:\yhu39\github\glycoproteinnotebook-private\public_data\matrix\PDAC_G_PDC000272"
path = os.path.join(wd, "DIG_nglycoform-peptide_matrix-abundances-MD_norm.tsv")
df = pd.read_csv(path ,sep="\t",index_col=[0,1,2,3])

In [43]:
df.head(2)

Intensity.Reference  \
Site                                    Gene  Sequence   Glycan                            
ENSP00000353032@196;ENSP00000336607@180 P2RX4 AAENFTLLVK N2H6F0S0G0            15.364905   
                                                         N2H7F0S0G0            16.658094   

                                                                       pool_01  \
Site                                    Gene  Sequence   Glycan                  
ENSP00000353032@196;ENSP00000336607@180 P2RX4 AAENFTLLVK N2H6F0S0G0  15.364905   
                                                         N2H7F0S0G0  16.658094   

                                                                      QC1_C_01  \
Site                                    Gene  Sequence   Glycan                  
ENSP00000353032@196;ENSP00000336607@180 P2RX4 AAENFTLLVK N2H6F0S0G0  15.946638   
                                                         N2H7F0S0G0  17.848041   

                                                                     C3N-03884_T_01  \
Site                                    Gene  Sequence   Glycan                       
ENSP00000353032@196;ENSP00000336607@180 P2RX4 AAENFTLLVK N2H6F0S0G0       16.043294   
                                                         N2H7F0S0G0       17.354672   

                                                                     C3L-00589_N_F_01  \
Site                                    Gene  Sequence   Glycan                         
ENSP00000353032@196;ENSP00000336607@180 P2RX4 AAENFTLLVK N2H6F0S0G0         15.815325   
                                                         N2H7F0S0G0         15.980933   

                                                                     C3L-03123_N_01  \
Site                                    Gene  Sequence   Glycan                       
ENSP00000353032@196;ENSP00000336607@180 P2RX4 AAENFTLLVK N2H6F0S0G0       15.782230   
                                                         N2H7F0S0G0       16.870973   

                                                                     C3L-01687_N_F_01  \
Site                                    Gene  Sequence   Glycan                         
ENSP00000353032@196;ENSP00000336607@180 P2RX4 AAENFTLLVK N2H6F0S0G0         15.413388   
                                                         N2H7F0S0G0         16.246879   

                                                                     C3L-03123_T_01  \
Site                                    Gene  Sequence   Glycan                       
ENSP00000353032@196;ENSP00000336607@180 P2RX4 AAENFTLLVK N2H6F0S0G0       16.336067   
                                                         N2H7F0S0G0       18.094625   

                                                                     C3L-01687_T_01  \
Site                                    Gene  Sequence   Glycan                       
ENSP00000353032@196;ENSP00000336607@180 P2RX4 AAENFTLLVK N2H6F0S0G0       15.675056   
                                                         N2H7F0S0G0       16.728724   

                                                                     C3L-00589_T_01  \
Site                                    Gene  Sequence   Glycan                       
ENSP00000353032@196;ENSP00000336607@180 P2RX4 AAENFTLLVK N2H6F0S0G0       15.879306   
                                                         N2H7F0S0G0       17.303349   

                                                                     ...  \
Site                                    Gene  Sequence   Glycan      ...   
ENSP00000353032@196;ENSP00000336607@180 P2RX4 AAENFTLLVK N2H6F0S0G0  ...   
                                                         N2H7F0S0G0  ...   

                                                                     C3L-03513_N_25  \
Site                                    Gene  Sequence   Glycan                       
ENSP00000353032@196;ENSP00000336607@180 P2RX4 AAENFTLLVK N2H6F0S0G0       14.881852   
                 

In [48]:
df2 = df.loc[:,tumor_samples].dropna()

In [49]:
df2.shape

(1017, 105)

In [56]:
from scipy.stats import variation
rows = []
for index,row in df2.iterrows():
    rows.append([variation([np.power(2,i) for i in list(row)])])
cv_df = pd.DataFrame(rows,columns=['cv'],index= df2.index)

glycopeptides = cv_df[cv_df['cv']>0.25].index

data2 = df2[df2.index.isin(glycopeptides)]
glycopeptides =  [f'{site}@{gene}@{seq}@{glycan}' for site,gene,seq,glycan in glycopeptides]
data2.index = glycopeptides
tumor_expression_path = os.path.join(job_dir,'expression_data.tsv')
data2.to_csv(tumor_expression_path,sep='\t',index=True)

Extract tumor samples from glycopeptide expression data based on pathological status,

calculates the coefficient of variation (CV) for each glycopeptide, selects glycopeptides with CV greater than 0.25.

Map glcopeptides with cv>0.25 in tumor patients with glycan type.

In [57]:
# left annotation
from gpnotebook.tools.glycan import decide_glycan_type

glycan_type_map = dict(zip(glycopeptides,[decide_glycan_type(i) for i in glycopeptides]))
  
left_ann_data_path =  os.path.join(job_dir,'left_annotation_data.tsv')
rows = []
for i in glycan_type_map:
    rows.append([i,glycan_type_map[i]])
left_ann_data = pd.DataFrame(rows,columns=['Glycopeptide','GlycanType'])
left_ann_data.to_csv(left_ann_data_path,sep="\t",index=False)

In [14]:
left_ann_data

,Glycopeptide,GlycanType
0,APOD|ENSG00000189058.9@ENSP00000345179.3@98@AD...,F+S
1,POSTN|ENSG00000133110.15@ENSP00000437959.1@599...,F+S
2,PLXNB2|ENSG00000196576.15@ENSP00000409171.1@10...,HM
3,HSPG2|ENSG00000142798.20@ENSP00000363827.3@378...,only_F
4,GUSB|ENSG00000169919.17@ENSP00000302728.4@272@...,HM
...,...,...
1457,STT3B|ENSG00000163527.10@ENSP00000295770.2@623...,HM
1458,ABI3BP|ENSG00000154175.17@ENSP00000420524.1@31...,F+S
1459,CELA3B|ENSG00000219073.8@ENSP00000338369.6@114...,only_F
1460,MFGE8|ENSG00000140545.15@ENSP00000268150.8@238...,HM


Map glycan types with colors.

In [58]:

# left annotation settings, including color, order
left_ann_settings_path = os.path.join(job_dir,'left_annotation_settings.yml')
left_ann_settings = {
    "glycan_type_index" :{
    "HM": 1,
    "only_F":2,
    "only_S":3,
    "F+S":4,
    "Other":5
    },
    "glycan_type_color" : {
        "HM": 'green',
    "only_F": 'red',
    "only_S": 'purple',
    "F+S": 'orange',
    "Other": 'grey'
}
}
with open(left_ann_settings_path,'w') as f:
    yaml.dump(left_ann_settings,f,default_flow_style=False)
    

Parameters for NMF clustering.

In [80]:
nmf_parameters_path = os.path.join(job_dir, 'nmf_parameters.yml')
nmf_parameters = {
    'k_range': {
        'min': 3,
        'max': 5,
    },
    'test':{
        'nruns': 50
    },
    'opt_k':{
        'nruns': 500,
        'predefined': 0,
        'value': 4,
        'feature_prob': 0.8
    }
}
with open(nmf_parameters_path,'w') as f:
    yaml.dump(nmf_parameters,f,default_flow_style=False)

Generate a YAML configuration file (nmf_configs.yml) containing paths to various data required for NMF clustering.

In [60]:
config_data = {
    'input': {
        'expression_data': tumor_expression_path,
        'left_annotation_data': left_ann_data_path ,
        'left_annotation_settings': left_ann_settings_path,
        'top_annotation_data': top_ann_data_path,
        'top_annotatin_settings': top_ann_settings_path,
        'nmf_parameters': nmf_parameters_path
    },
    'output':{
        'out_dir': job_dir
    }
}
nmf_configs_path = os.path.join(job_dir,'nmf_configs.yml')
with open(nmf_configs_path,'w') as f:
    yaml.dump(config_data,f,default_flow_style=False)